In [5]:

import duckdb
con = duckdb.connect()

F = "../tests/fixtures"

def q(sql):
    return con.execute(sql).fetchone()[0]

In [6]:
print("brandstof zonder voertuig   :", q(f"""
  select count(*) from read_parquet('{F}/rdw_brandstof.parquet') b
  where not exists (select 1 from read_parquet('{F}/rdw_gekentekende_voertuigen.parquet') v
                    where v.kenteken = b.kenteken)"""))

brandstof zonder voertuig   : 0


In [7]:
print("constatering zonder voertuig:", q(f"""
  select count(*) from read_parquet('{F}/rdw_geconstateerde_gebreken.parquet') g
  where not exists (select 1 from read_parquet('{F}/rdw_gekentekende_voertuigen.parquet') v
                    where v.kenteken = g.kenteken)"""))

constatering zonder voertuig: 0


In [8]:
print("constatering zonder code    :", q(f"""
  select count(*) from read_parquet('{F}/rdw_geconstateerde_gebreken.parquet') g
  where not exists (select 1 from read_parquet('{F}/rdw_gebreken.parquet') d
                    where d.gebrek_identificatie = g.gebrek_identificatie)"""))

constatering zonder code    : 0


In [9]:
print("dubbele constatering        :", q(f"""
  select count(*) from (
    select 1 from read_parquet('{F}/rdw_geconstateerde_gebreken.parquet')
    group by kenteken, meld_datum_door_keuringsinstantie,
             meld_tijd_door_keuringsinstantie, gebrek_identificatie
    having count(*) > 1)"""))

dubbele constatering        : 1


In [10]:
print("meldtijden van 3 tekens     :", q(f"""
  select count(*) from read_parquet('{F}/rdw_geconstateerde_gebreken.parquet')
  where length(meld_tijd_door_keuringsinstantie) = 3"""))

meldtijden van 3 tekens     : 1930
